In [59]:
# Preparation
import pandas as pd, numpy as np, sqlalchemy as sqla
import xlrd, pdfplumber, shutil, decouple
from sqlalchemy import String
from IPython.display import display

engine = sqla.create_engine(decouple.config("MIS_DB"))
mis_file_path = decouple.config("MIS_FILE_PATH")

pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', 100)
pd.set_option("display.float_format", "{:.2f}".format)

month_num_arr = {
    "January": "01", "February": "02", "March": "03", "April": "04", "May": "05", "June": "06",
    "July": "07", "August": "08", "September": "09", "October": "10", "November": "11", "December": "12"
}

col_order = ["year", "month", "account_name", "store_name", "store_code", "item_desc", "item_code", "qty", "amount", "net_amount"]

In [60]:
# Waltermart Supermarket Inc.

def clean_wm(file_name):
    wm_df = pd.read_csv(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\{file_name}", header=None)

    wm_df = wm_df.drop(0, axis=0)

    wm_df[0] = "Waltermart Supermarket Inc."
    wm_df[7] = pd.to_datetime(wm_df[7])
    wm_df[9] = wm_df[9].str.title()
    wm_df[19] = wm_df[7].dt.year
    wm_df[20] = wm_df[7].dt.month_name()

    month_num = wm_df.iloc[0, 7].month

    wm_df = wm_df.rename(columns={
        0: "account_name", 8: "store_code", 9: "store_name", 10: "item_code", 12: "item_desc",
        14: "qty", 15: "amount", 17: "net_amount", 19: "year", 20: "month"
    })
    wm_df = wm_df[col_order]

    shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Waltermart.csv", 
        f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled WM Extracted\\{wm_df.iloc[0, 0]} {str(month_num).zfill(2)} {wm_df.iloc[0, 1]}.csv")

    return wm_df

In [61]:
# LCC Supermarket

def clean_lcc_smkt(file_name):
    lcc_xls = xlrd.open_workbook(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\{file_name}", ignore_workbook_corruption=True)
    lcc_smkt_xl = pd.read_excel(lcc_xls, sheet_name=None, header=None)
    lcc_smkt = pd.concat(lcc_smkt_xl.values(), ignore_index=True)

    lcc_smkt[1] = lcc_smkt[1].str.strip()
    lcc_smkt[6] = "LCC Supermarket"
    lcc_smkt[7] = lcc_smkt[2].str.extract(r"Store\s+:\s+\d+-(\D+)")
    lcc_smkt[7] = lcc_smkt[7].str.strip()
    lcc_smkt[8] = lcc_smkt[2].str.extract(r"Store\s+:\s+(\d+)-\D+")
    lcc_smkt[9] = lcc_smkt[0].str.extract(r"Period\s+:\s+(\D+) ")
    lcc_smkt[10] = np.where(lcc_smkt[0].str.contains("Period"), lcc_smkt[0].str[-4:], np.nan)

    for x in range(6, 11, 1):
        lcc_smkt[x] = lcc_smkt[x].ffill()

    lcc_smkt = lcc_smkt[lcc_smkt[1].notna() & ~(lcc_smkt[1] == "DESCRIPTION")]
    lcc_smkt[11] = lcc_smkt[5].astype(int) * 0.75

    lcc_smkt = lcc_smkt.rename(columns={
        0: "item_code", 1: "item_desc", 4: "qty", 5: "amount", 6: "account_name",
        7: "store_name", 8: "store_code", 9: "month", 10: "year", 11: "net_amount" 
    })

    lcc_smkt = lcc_smkt[col_order]

    shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\LCC SMKT.xls", 
        f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled LCC SMKT Extracted\\{lcc_smkt.iloc[0, 0]} {month_num_arr.get(lcc_smkt.iloc[0, 1])} {lcc_smkt.iloc[0, 1]}.xls")
    
    return lcc_smkt

In [62]:
# LCC Department Store

def clean_lcc_ds(file_name):
    lcc_xls = xlrd.open_workbook(
        f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\{file_name}",
        ignore_workbook_corruption=True
    )

    lcc_ds_xl = pd.read_excel(lcc_xls, sheet_name=None, header=None)
    lcc_ds = pd.concat(lcc_ds_xl.values(), ignore_index=True)

    lcc_ds[1] = lcc_ds[1].str.strip()
    lcc_ds[6] = "LCC Department Store"
    lcc_ds[7] = lcc_ds[2].str.extract(r"Store\s+:\s+\d+-(\D+)")
    lcc_ds[7] = lcc_ds[7].str.strip()
    lcc_ds[8] = lcc_ds[2].str.extract(r"Store\s+:\s+(\d+)-\D+")
    lcc_ds[9] = lcc_ds[0].str.extract(r"Period\s+:\s+(\D+) ")
    lcc_ds[10] = np.where(lcc_ds[0].str.contains("Period"), lcc_ds[0].str[-4:], np.nan)

    for x in range(6, 11, 1):
        lcc_ds[x] = lcc_ds[x].ffill()

    lcc_ds = lcc_ds[lcc_ds[1].notna() & ~(lcc_ds[1] == "DESCRIPTION")]
    lcc_ds[11] = lcc_ds[5].astype(int) * 0.75

    lcc_ds = lcc_ds.rename(columns={
        0: "item_code", 1: "item_desc",  4: "qty", 5: "amount", 6: "account_name",
        7: "store_name", 8: "store_code", 9: "month", 10: "year", 11: "net_amount"
    })

    lcc_ds = lcc_ds[col_order]

    shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\LCC DS.xls", 
        f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled LCC DS Extracted\\{lcc_ds.iloc[0, 0]} {month_num_arr.get(lcc_ds.iloc[0, 1])} {lcc_ds.iloc[0, 1]}.xls")
    
    return lcc_ds

In [63]:
# Metro Gaisano

def clean_mg(file_name):
    
    def extract_data(path, texts):
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                lines = text.split('\n')
                for x in range(len(lines)):
                    line = lines[x]
                    texts.append(line)

    texts = []
    extract_data(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\{file_name}", texts)

    mg_df = pd.DataFrame(texts)

    # include_conditions = [
    #     mg_df[0].str.contains("SKU wise summary:"),
    #     mg_df[0].str.contains("SKU and transaction date wise summary:")
    # ]

    mg_df[1] = pd.to_datetime(mg_df[0].str.extract(r"^FromPeriod:\s+(\d{2}-\D+-\d{2})")[0], format="%d-%b-%y").dt.month_name()
    mg_df[2] = pd.to_datetime(mg_df[0].str.extract(r"^FromPeriod:\s+(\d{2}-\D+-\d{2})")[0], format="%d-%b-%y").dt.year
    mg_df[3] = "Metro Gaisano"
    mg_df[4] = mg_df[0].str.extract(r"Store:\s+\d+-(\D+)", expand=False).str.title()
    mg_df[5] = mg_df[0].str.extract(r"Store:\s+(\d+)-\D+")

    for x in range(1, 6, 1):
        mg_df[x] = mg_df[x].ffill()

    mg_df[6] = mg_df[0].str.extract(r"\d+\s+(\d+)\s+.+\s+\d+\s+[\d,]+\.\d+\s+[\d,]+\.\d+\s+[\d,]+\.\d+")
    mg_df[7] = mg_df[0].str.extract(r"\d+\s+\d+\s+(.+)\s+\d+\s+[\d,]+\.\d+\s+[\d,]+\.\d+\s+[\d,]+\.\d+")
    mg_df[8] = mg_df[0].str.extract(r"\d+\s+\d+\s+.+\s+(\d+)\s+[\d,]+\.\d+\s+[\d,]+\.\d+\s+[\d,]+\.\d+")
    mg_df[9] = mg_df[0].str.extract(r"\d+\s+\d+\s+.+\s+\d+\s+([\d,]+\.\d+)\s+[\d,]+\.\d+\s+[\d,]+\.\d+")
    mg_df[10] = mg_df[0].str.extract(r"\d+\s+\d+\s+.+\s+\d+\s+[\d,]+\.\d+\s+[\d,]+\.\d+\s+([\d,]+\.\d+)")
    mg_df[11] = np.where(
        mg_df[7].shift(-1).isna() & ~(mg_df[0].shift(-1).str.contains(("Total Amount|Report Date"), na=False, case=False)), 
        mg_df[7].astype(str) + " " + mg_df[0].shift(-1).astype(str),
        mg_df[7].astype(str)
    )

    mg_df = mg_df[~(mg_df[8].isna())]
    mg_df = mg_df.rename(columns={
        1: "month", 2: "year", 3: "account_name", 4: "store_name", 5: "store_code", 6: "item_code",
        8: "qty", 9: "net_amount", 10: "amount", 11: "item_desc"
    })

    mg_df = mg_df[col_order]

    shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Metro Gaisano.pdf", 
        f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled Metro Gaisano Extracted\\{str(mg_df.iloc[0, 0])[:4]} {month_num_arr.get(mg_df.iloc[0, 1])} {mg_df.iloc[0, 1]}.pdf")
    
    return mg_df

In [64]:
# Ever Supermarket

def clean_ever(file_name1, file_name2):
    
    def extract_data(path):
        texts = []
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                lines = text.split('\n')
                texts.extend(lines)
        return texts

    ever1 = extract_data(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\{file_name1}")
    ever2 = extract_data(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\{file_name2}")

    ever_df = pd.DataFrame(ever1 + ever2, columns=[0])

    ever_df[1] = ever_df[0].str.extract(r"\w+.+?\s+[A-Za-z]+\s+\d{2},\s+(\d{4})\s+\w+\s+\d{2},\s+\d{4}\s+-?[\d,]+\.\d+")
    ever_df[2] = ever_df[0].str.extract(r"\w+.+?\s+([A-Za-z]+)\s+\d{2},\s+\d{4}\s+\w+\s+\d{2},\s+\d{4}\s+-?[\d,]+\.\d+")
    ever_df[3] = ever_df[0].str.extract(r"\w+(.+?)\s+[A-Za-z]+\s+\d{2},\s+\d{4}\s+\w+\s+\d{2},\s+\d{4}\s+-?[\d,]+\.\d+")
    ever_df[4] = ever_df[0].str.extract(r"(\w+).+?\s+[A-Za-z]+\s+\d{2},\s+\d{4}\s+\w+\s+\d{2},\s+\d{4}\s+-?[\d,]+\.\d+")

    for x in range(1, 5, 1):
        ever_df[x] = ever_df[x].ffill()

    ever_df[5] = ever_df[0].str.extract(r"^\d+\s+\d+\s+(\d+)\s+.+\s+.+\s+.+$")
    ever_df[6] = ever_df[0].str.extract(r"^\d+\s+\d+\s+\d+\s+(.+)\s+.+\s+.+$")
    ever_df[7] = ever_df[0].str.extract(r"^\d+\s+\d+\s+\d+\s+.+\s+(.+)\s+.+$")
    ever_df[8] = ever_df[0].str.extract(r"^\d+\s+\d+\s+\d+\s+.+\s+.+\s+(.+)$")
    ever_df[9] = ever_df[8].astype(float) * 0.75
    ever_df[10] = "Ever Supermarket"

    ever_df[11] = np.where(
        ever_df[6].shift(-1).isna()
        & ~(
            ever_df[0].shift(-1).str.match(r"\d{2}-\d{2}-\d{4}\s\d+\.\d+\s\d+\.\d+")
            | ever_df[0].shift(-1).str.match(r"Total:\s+\w+")
            | ever_df[0].shift(-1).str.match(r"\d{2}-\d{2}-\d{4}\s+\d+\s+\d+\.\d+")
            | ever_df[0].shift(-1).str.match(r"\w+.+?\s+[A-Za-z]+\s+\d{2},\s+\d{4}\s+\w+\s+\d{2},\s+\d{4}\s+-?[\d,]+\.\d+")
            | ever_df[0].shift(-1).str.contains(r"This is a system-generated advice\. No signature is required\.\s*", regex=True)
        ),
        ever_df[6].astype(str) + " " + ever_df[0].shift(-1).astype(str),
        ever_df[6].astype(str)
    )

    ever_df = ever_df[~(ever_df[8].isna())]
    ever_df = ever_df.drop([0, 6], axis=1)
    ever_df = ever_df.rename(columns={
        1: "year", 2: "month", 3: "store_name", 4: "store_code", 5: "item_code",
        7: "qty", 8: "amount", 9: "net_amount", 10: "account_name", 11: "item_desc"
    })

    ever_df = ever_df[col_order]

    shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Ever SMKT 1.pdf", 
        f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled Ever SMKT Extracted\\{ever_df.iloc[0, 0]} {month_num_arr.get(ever_df.iloc[0, 1])} {ever_df.iloc[0, 1]} 1.pdf")

    shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Ever SMKT 2.pdf", 
        f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled Ever SMKT Extracted\\{ever_df.iloc[0, 0]} {month_num_arr.get(ever_df.iloc[0, 1])} {ever_df.iloc[0, 1]} 2.pdf")
    
    return ever_df
    

In [65]:
# Consignment Portal DataFrames
print("Waltermart")
wm_df = clean_wm("Waltermart.csv")
display(wm_df.head(5))

print("LCC Supermarket")
lcc_smkt = clean_lcc_smkt("LCC SMKT.xls")
display(lcc_smkt.head(5))

print("LCC Department Store")
lcc_ds = clean_lcc_ds("LCC DS.xls")
display(lcc_ds.head(5))

print("Metro Gaisano")
mg_df = clean_mg("Metro Gaisano.pdf")
display(mg_df.head(5))

# print("Ever Supermarket")
# ever_df = clean_ever("Ever SMKT 1.pdf", "Ever SMKT 2.pdf")
# display(ever_df.head(5))

# Combine selected dataframes

# all dataframes
# [wm_df, lcc_smkt, lcc_ds, mg_df, ever_df]

cn_sin = pd.concat([wm_df, lcc_smkt, lcc_ds, mg_df], ignore_index=True)

cn_sin["qty"] = pd.to_numeric(cn_sin["qty"].astype(str).str.replace(",", ""), errors="coerce")
cn_sin["amount"] = pd.to_numeric(cn_sin["amount"].astype(str).str.replace(",", ""), errors="coerce")
cn_sin["net_amount"] = pd.to_numeric(cn_sin["net_amount"].astype(str).str.replace(",", ""), errors="coerce")

cn_sin = cn_sin.astype({
    "year": int, "month": str, "account_name": str, "store_name": str,
    "store_code": str, "item_desc": str, "item_code": str
})

cn_sin.to_sql(
    name="cn_prtl_mtd", con=engine, schema="staging", if_exists="replace", index=False, 
    dtype={col: String(255) for col in cn_sin.select_dtypes(include="object").columns}
)

print(cn_sin.shape)
print(cn_sin.dtypes)

Waltermart


,year,month,account_name,store_name,store_code,item_desc,item_code,qty,amount,net_amount
1,2026,June,Waltermart Supermarket Inc.,Waltermart-Makati,402,IWHITE TNR AQUA PUREGEL 85ML/1,346396,4,820,656
2,2026,June,Waltermart Supermarket Inc.,Waltermart-Makati,402,IWHITE TNR AQUA PURE GEL 8ML/1,346395,6,132,105.6
3,2026,June,Waltermart Supermarket Inc.,Waltermart-Makati,402,IWHITE FCRM AQUA MST GLW 6ML/1,346394,31,837,669.6
4,2026,June,Waltermart Supermarket Inc.,Waltermart-Makati,402,IWHITE FCRM AQUA MSTGLW 50ML/1,346393,6,1494,1195.2
5,2026,June,Waltermart Supermarket Inc.,Waltermart-Makati,402,IWHT FCRM BBHLC LYT 4ML/1,333013,17,459,367.2


LCC Supermarket


,year,month,account_name,store_name,store_code,item_desc,item_code,qty,amount,net_amount
7,2026,June,LCC Supermarket,Tabaco Supermarket TCM,137,iWhiteNosePack3.5g,3866282,16,320,240.00
8,2026,June,LCC Supermarket,Tabaco Supermarket TCM,137,iWhiteWhiteningPack8g,3866283,10,250,187.50
9,2026,June,LCC Supermarket,Tabaco Supermarket TCM,137,iWhiteSkinWhtnngVitaFCream65ml,3866289,1,239,179.25
10,2026,June,LCC Supermarket,Tabaco Supermarket TCM,137,iWhiteSknWhtnngVitaAquaMCr50ml,3866291,1,239,179.25
11,2026,June,LCC Supermarket,Tabaco Supermarket TCM,137,IwhiteBBHolicLight4ml,4155456,8,216,162.00


LCC Department Store


,year,month,account_name,store_name,store_code,item_desc,item_code,qty,amount,net_amount
7,2026,June,LCC Department Store,Tabaco Department Store,100,iWhiteNosePack3.5g,3866282,10,200,150.00
8,2026,June,LCC Department Store,Tabaco Department Store,100,iWhiteWhiteningPack8g,3866283,3,75,56.25
9,2026,June,LCC Department Store,Tabaco Department Store,100,iWhiteSkinWhtnngVitaFCream6ml,3866284,56,1400,1050.00
10,2026,June,LCC Department Store,Tabaco Department Store,100,iWhiteSknWhtnngVitaMoistFW10ml,3866285,3,69,51.75
11,2026,June,LCC Department Store,Tabaco Department Store,100,IwhiteBBHolicLight4ml,4155456,37,999,749.25


Metro Gaisano


,year,month,account_name,store_name,store_code,item_desc,item_code,qty,amount,net_amount
925,2026.00,June,Metro Gaisano,Metro Gaisano (Colon),2001,IWHITE BB HOLIC LIGHT,102292769,37,699.30,624.38
926,2026.00,June,Metro Gaisano,Metro Gaisano (Colon),2001,IWHITE BB HOLIC BEIGE,102292784,25,472.50,421.88
927,2026.00,June,Metro Gaisano,Metro Gaisano (Colon),2001,IWHITE BB HOLIC LIGHT TUBE 25ML,102403805,5,766.50,684.38
928,2026.00,June,Metro Gaisano,Metro Gaisano (Colon),2001,IWHITE BB HOLIC BEIGE TUBE 25ML,102403809,4,613.20,547.50
929,2026.00,June,Metro Gaisano,Metro Gaisano (Colon),2001,IWHITE THE ORIGINAL NOSE PACK,102448588,2,28.00,25.00


(1625, 10)
year              int64
month            object
account_name     object
store_name       object
store_code       object
item_desc        object
item_code        object
qty               int64
amount          float64
net_amount      float64
dtype: object


In [66]:
# Create a checker file

qry = """
SELECT
	cn.account_name,
	ad.account_chain, 
    cn.item_code, 
    cn.item_desc,
    ic.product_code
FROM staging.cn_prtl_mtd AS cn

LEFT JOIN ref.account_details AS ad
	ON cn.account_name = ad.account_name
    
LEFT JOIN ref.item_codes AS ic
	ON ad.account_chain = ic.account_chain
    AND cn.item_code = ic.item_code

WHERE ic.product_code IS NULL;
"""

df = pd.read_sql_query(qry, con=engine)
df.to_csv("C:\\MIS Outputs\\Consignment Portal Checker.csv", index=False)